# 🐍 Python OOP — Intermediate Interview Prep

| # | Topic |
|---|-------|
| 1 | Classes & Objects |
| 2 | Encapsulation |
| 3 | Inheritance |
| 4 | Polymorphism |
| 5 | Abstraction |
| 6 | Dunder / Magic Methods |
| 7 | Class & Static Methods |
| 8 | Design Patterns & Advanced |

---
# 🏗️ 1. Classes & Objects

## Q1. Class vs Object

In [ ]:
# Class — blueprint / template (defines structure and behavior)
class Car:
    def __init__(self, brand, speed):
        self.brand = brand
        self.speed = speed

    def drive(self):
        return f"{self.brand} driving at {self.speed} km/h"

# Object — instance of a class (actual data in memory)
car1 = Car('Toyota', 120)   # object 1
car2 = Car('BMW', 200)      # object 2

print(car1.drive())  # Toyota driving at 120 km/h
print(car2.drive())  # BMW driving at 200 km/h

# Each object has its own data but shares the class methods
print(type(car1))           # <class '__main__.Car'>
print(isinstance(car1, Car))# True
print(id(car1) == id(car2)) # False — different memory addresses

# Class = blueprint (one per program)
# Object = instance (many per class, each with own data)

## Q2. `__init__` — Is it a constructor?

In [ ]:
# __init__ is an INITIALIZER, not a true constructor.
# The real constructor is __new__ (which creates the object).
# __init__ just initializes the already-created object.

class Person:
    def __init__(self, name, age):   # called after object is created
        self.name = name             # sets instance attributes
        self.age = age
        print(f"Person {name} initialized")

p = Person('Alice', 30)
# Output: Person Alice initialized

# What actually happens:
# 1. Python calls Person.__new__(Person)  → creates the object in memory
# 2. Python calls Person.__init__(obj, 'Alice', 30) → initializes attributes

# __init__ rules:
# - Always takes self as first argument
# - Never returns a value (returns None implicitly)
# - Can be omitted if no initialization needed

class Empty:
    pass  # valid class with no __init__

e = Empty()  # works fine

## Q3. Instance Variables vs Class Variables

In [ ]:
class Employee:
    company = 'TechCorp'      # Class variable — shared by ALL instances
    employee_count = 0        # Class variable — tracks total employees

    def __init__(self, name, salary):
        self.name = name       # Instance variable — unique per object
        self.salary = salary   # Instance variable — unique per object
        Employee.employee_count += 1  # modify class variable via class name

e1 = Employee('Alice', 50000)
e2 = Employee('Bob', 60000)

print(e1.name)              # Alice  (instance variable)
print(e2.name)              # Bob    (instance variable)
print(e1.company)           # TechCorp (class variable)
print(e2.company)           # TechCorp (same class variable)
print(Employee.employee_count)  # 2

# Changing class variable affects ALL instances
Employee.company = 'NewCorp'
print(e1.company)           # NewCorp
print(e2.company)           # NewCorp

# ⚠️ Gotcha — assigning via instance creates a NEW instance variable!
e1.company = 'StartupCo'   # creates instance variable on e1 only!
print(e1.company)           # StartupCo  (instance variable shadows class var)
print(e2.company)           # NewCorp    (still using class variable)
print(Employee.company)     # NewCorp    (class variable unchanged)

# ⚠️ Mutable class variables are dangerous
class BadExample:
    items = []  # shared list — ALL instances share this!

    def add(self, item):
        self.items.append(item)  # modifies the SHARED list!

a = BadExample()
b = BadExample()
a.add('x')
print(b.items)  # ['x'] — b sees a's item!

# Fix — use instance variable for mutable defaults
class GoodExample:
    def __init__(self):
        self.items = []  # each instance gets its own list

## Q4. What is `self` and why is it needed?

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name      # 'self' refers to the current instance

    def bark(self):
        return f"{self.name} says Woof!"

dog = Dog('Rex')
print(dog.bark())  # Rex says Woof!

# What Python does internally:
# dog.bark()  →  Dog.bark(dog)  (self = dog)

# 'self' is NOT a keyword — it's just a convention
# You can name it anything (but don't!)
class Weird:
    def greet(this):         # 'this' works but is bad practice
        return "Hello"

# Why self is needed:
# Python methods are just functions defined inside a class
# Without self, there's no way to access the specific instance's data

class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1   # 'self.count' → this instance's count

c1 = Counter()
c2 = Counter()
c1.increment()
c1.increment()
print(c1.count)  # 2
print(c2.count)  # 0 — independent!

## Q5. `__str__` vs `__repr__`

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # Human-readable — for end users
        # Called by: print(), str(), f-strings
        return f"Point({self.x}, {self.y})"

    def __repr__(self):
        # Unambiguous — for developers/debugging
        # Called by: repr(), interactive console, logging
        # Should ideally return string that can recreate the object
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 4)

print(str(p))   # Point(3, 4)         → uses __str__
print(repr(p))  # Point(x=3, y=4)     → uses __repr__
print(p)        # Point(3, 4)         → uses __str__
print(f"{p}")   # Point(3, 4)         → uses __str__
print(f"{p!r}") # Point(x=3, y=4)    → forces __repr__

# If only __repr__ is defined, it's used for both str() and repr()
# If only __str__ is defined, repr() falls back to default <__main__.Point object>

# Rule of thumb:
# __str__  → pretty, readable (for print)
# __repr__ → precise, unambiguous (for debug)
# Always define __repr__ at minimum

class NoStr:
    def __repr__(self):
        return "NoStr()"

n = NoStr()
print(n)        # NoStr()  ← falls back to __repr__
print(repr(n))  # NoStr()

---
# 🔒 2. Encapsulation

## Q6. Public, Protected, and Private Attributes

In [ ]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner          # Public    — accessible anywhere
        self._balance = balance     # Protected — convention: don't touch outside class/subclass
        self.__pin = 1234           # Private   — name mangled, hard to access outside class

    def get_balance(self):
        return self._balance        # access protected inside class — fine

    def verify_pin(self, pin):
        return self.__pin == pin    # access private inside class — fine

acc = BankAccount('Alice', 5000)

# Public — fully accessible
print(acc.owner)          # Alice ✅

# Protected — accessible but conventionally private
print(acc._balance)       # 5000 ✅ (works but shouldn't be done outside class)

# Private — name mangled, not directly accessible
# print(acc.__pin)        # ❌ AttributeError!
print(acc._BankAccount__pin)  # 1234 ✅ (name mangling: _ClassName__attr)

# Summary:
# self.attr    → Public    → accessible everywhere
# self._attr   → Protected → convention only, accessible but shouldn't be
# self.__attr  → Private   → name mangled to _ClassName__attr

## Q7. Name Mangling (`__var`)

In [ ]:
class Parent:
    def __init__(self):
        self.__secret = 'parent secret'   # mangled to _Parent__secret

    def reveal(self):
        return self.__secret              # works inside class

class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__secret = 'child secret'    # mangled to _Child__secret (different!)

    def reveal_child(self):
        return self.__secret              # returns 'child secret'

c = Child()
print(c.reveal())          # parent secret  ← Parent's __secret
print(c.reveal_child())    # child secret   ← Child's __secret

# They don't clash because of name mangling:
print(c._Parent__secret)   # parent secret
print(c._Child__secret)    # child secret

# Why name mangling exists:
# To prevent accidental override in subclasses
# Not for true security — still accessible via _ClassName__attr

# Check all attributes
print([a for a in dir(c) if 'secret' in a])
# ['_Child__secret', '_Parent__secret']

## Q8. `@property`, `@setter`, `@deleter`

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius   # private backing attribute

    @property
    def celsius(self):
        # Getter — called when you READ the attribute
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        # Setter — called when you WRITE the attribute
        if value < -273.15:
            raise ValueError("Temperature below absolute zero!")
        self._celsius = value

    @celsius.deleter
    def celsius(self):
        # Deleter — called when you DELETE the attribute
        print("Deleting temperature")
        del self._celsius

    @property
    def fahrenheit(self):
        # Computed / derived property — no setter needed
        return (self._celsius * 9/5) + 32


t = Temperature(25)
print(t.celsius)       # 25    ← calls getter
print(t.fahrenheit)    # 77.0  ← computed property

t.celsius = 30         # calls setter with validation
print(t.celsius)       # 30

# t.celsius = -300   # ❌ ValueError: Temperature below absolute zero!

del t.celsius          # calls deleter

# Why use @property instead of direct attributes?
# 1. Add validation without changing the public API
# 2. Compute derived values on the fly
# 3. Control read/write/delete access independently
# 4. Backward compatible — callers use t.celsius not t.get_celsius()

# Read-only property (no setter)
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        import math
        return math.pi * self.radius ** 2

c = Circle(5)
print(c.area)     # 78.53...
# c.area = 100    # ❌ AttributeError: can't set attribute

---
# 🧬 3. Inheritance

## Q9. Single vs Multiple Inheritance

In [ ]:
# Single Inheritance — one parent
class Animal:
    def breathe(self):
        return "Breathing"

class Dog(Animal):           # Dog inherits from Animal
    def bark(self):
        return "Woof!"

d = Dog()
print(d.breathe())  # Breathing — inherited
print(d.bark())     # Woof!


# Multiple Inheritance — more than one parent
class Flyable:
    def fly(self):
        return "Flying"

class Swimmable:
    def swim(self):
        return "Swimming"

class Duck(Animal, Flyable, Swimmable):  # inherits from 3 classes
    def quack(self):
        return "Quack!"

duck = Duck()
print(duck.fly())     # Flying
print(duck.swim())    # Swimming
print(duck.breathe()) # Breathing


# ⚠️ Diamond Problem — the main issue with multiple inheritance
class A:
    def hello(self):
        return "Hello from A"

class B(A):
    def hello(self):
        return "Hello from B"

class C(A):
    def hello(self):
        return "Hello from C"

class D(B, C):   # D inherits from both B and C (which both inherit A)
    pass

#     A
#    / \
#   B   C
#    \ /
#     D    ← which hello() does D use? B or C?

d = D()
print(d.hello())  # Hello from B  ← Python uses MRO (left to right)
print(D.__mro__)  # D → B → C → A → object

## Q10. MRO — Method Resolution Order

In [ ]:
# Python uses C3 Linearization algorithm to determine MRO
# Rule: left to right, depth first, respecting inheritance order

class A:
    def who(self): return 'A'

class B(A):
    def who(self): return 'B'

class C(A):
    def who(self): return 'C'

class D(B, C):
    pass

# MRO for D:
print(D.__mro__)
# (<class 'D'>, <class 'B'>, <class 'C'>, <class 'A'>, <class 'object'>)

# So D.who() → B.who() → returns 'B'
print(D().who())  # B

# More complex MRO example
class X: pass
class Y: pass
class Z: pass
class A(X, Y): pass
class B(Y, Z): pass
class M(A, B, Z): pass

print(M.__mro__)
# M → A → X → B → Y → Z → object

# Use mro() method
print([c.__name__ for c in M.mro()])

# ⚠️ Inconsistent MRO raises TypeError
# class Broken(A, B) where A and B have conflicting orders would fail

## Q11. `super()` — When and Why

In [ ]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        return f"Name: {self.name}, Age: {self.age}"

class Dog(Animal):
    def __init__(self, name, age, breed):
        super().__init__(name, age)  # calls Animal.__init__ — avoids code duplication
        self.breed = breed

    def info(self):
        base_info = super().info()   # calls Animal.info()
        return f"{base_info}, Breed: {self.breed}"

dog = Dog('Rex', 3, 'Labrador')
print(dog.info())  # Name: Rex, Age: 3, Breed: Labrador


# super() with multiple inheritance follows MRO
class A:
    def greet(self):
        print("Hello from A")

class B(A):
    def greet(self):
        print("Hello from B")
        super().greet()   # calls next in MRO (A)

class C(A):
    def greet(self):
        print("Hello from C")
        super().greet()   # calls next in MRO (A)

class D(B, C):
    def greet(self):
        print("Hello from D")
        super().greet()   # follows MRO: D → B → C → A

D().greet()
# Hello from D
# Hello from B
# Hello from C
# Hello from A
# super() cooperatively calls EACH class in MRO order!

# Why super() over Parent.__init__(self, ...):
# 1. Works correctly with multiple inheritance (MRO)
# 2. Avoids hardcoding parent class name
# 3. More maintainable when refactoring

## Q12. `isinstance()` vs `issubclass()`

In [ ]:
class Animal: pass
class Dog(Animal): pass
class Cat(Animal): pass

dog = Dog()

# isinstance(object, class) — checks if an OBJECT is an instance of a class
print(isinstance(dog, Dog))     # True  — dog is a Dog
print(isinstance(dog, Animal))  # True  — Dog inherits from Animal
print(isinstance(dog, Cat))     # False — dog is not a Cat

# isinstance with multiple types
print(isinstance(dog, (Cat, Dog)))  # True — is it Cat OR Dog?
print(isinstance(42, (int, float))) # True

# issubclass(class, class) — checks if a CLASS is a subclass of another CLASS
print(issubclass(Dog, Animal))   # True  — Dog inherits from Animal
print(issubclass(Dog, Dog))      # True  — class is subclass of itself
print(issubclass(Cat, Dog))      # False — Cat doesn't inherit from Dog
print(issubclass(Dog, object))   # True  — everything inherits from object

# Key difference:
# isinstance → works on INSTANCES (objects)
# issubclass → works on CLASSES

# Practical use
def process(animal):
    if isinstance(animal, Dog):
        print("Training a dog")
    elif isinstance(animal, Cat):
        print("Playing with a cat")

process(dog)  # Training a dog

## Q13. Method Overriding

In [ ]:
class Shape:
    def area(self):
        return 0

    def describe(self):
        return f"I am a shape with area {self.area()}"

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):                    # overrides Shape.area()
        import math
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):                    # overrides Shape.area()
        return self.w * self.h

c = Circle(5)
r = Rectangle(4, 6)

print(c.area())         # 78.53...
print(r.area())         # 24
print(c.describe())     # I am a shape with area 78.53...  ← uses overridden area()

# Calling parent method after overriding using super()
class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)  # calls Rectangle.__init__

    def area(self):
        base_area = super().area()    # calls Rectangle.area()
        print(f"[Square] Computed area via parent: {base_area}")
        return base_area

s = Square(4)
print(s.area())  # [Square] Computed area via parent: 16 → 16

---
# 🎭 4. Polymorphism

## Q14. Polymorphism — Real-World Example

In [ ]:
# Polymorphism = same interface, different behavior
# 'poly' = many, 'morph' = form

class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

class Cow:
    def speak(self):
        return "Moo!"

# Same function works on ALL types — that's polymorphism!
def animal_sounds(animals):
    for animal in animals:
        print(animal.speak())  # no type checking needed

animals = [Dog(), Cat(), Cow()]
animal_sounds(animals)
# Woof!
# Meow!
# Moo!

# Real-world: payment processing
class CreditCard:
    def pay(self, amount):
        return f"Paid ₹{amount} via Credit Card"

class UPI:
    def pay(self, amount):
        return f"Paid ₹{amount} via UPI"

class NetBanking:
    def pay(self, amount):
        return f"Paid ₹{amount} via Net Banking"

def checkout(payment_method, amount):
    print(payment_method.pay(amount))  # same interface, different behavior

checkout(CreditCard(), 500)   # Paid ₹500 via Credit Card
checkout(UPI(), 200)          # Paid ₹200 via UPI
checkout(NetBanking(), 1000)  # Paid ₹1000 via Net Banking

## Q15. Duck Typing

In [ ]:
# "If it walks like a duck and quacks like a duck, it's a duck"
# Python doesn't check the TYPE — it checks for the METHOD/ATTRIBUTE

class Duck:
    def quack(self): return "Quack!"
    def walk(self):  return "Waddle waddle"

class Person:
    def quack(self): return "I'm quacking like a duck!"
    def walk(self):  return "Walking like a duck!"

class RubberDuck:
    def quack(self): return "Squeak!"
    def walk(self):  return "Can't walk, I'm rubber"

def make_it_quack(duck):  # no type hint needed!
    print(duck.quack())   # just needs a quack() method

make_it_quack(Duck())        # Quack!
make_it_quack(Person())      # I'm quacking like a duck!
make_it_quack(RubberDuck())  # Squeak!
# Works for ALL three — Python doesn't care about the type


# Practical example — file-like objects
import io

def process_data(file_obj):
    # Works with any object that has .read()
    # Could be a real file, StringIO, BytesIO, network socket...
    data = file_obj.read()
    return data.upper()

# Works with real file:
# with open('data.txt') as f:
#     process_data(f)

# Works with StringIO (in-memory file):
fake_file = io.StringIO("hello world")
print(process_data(fake_file))  # HELLO WORLD

# Duck typing vs isinstance():
# isinstance() → checks TYPE  (explicit, rigid)
# duck typing  → checks BEHAVIOR (flexible, Pythonic)

## Q16. Operator Overloading

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):           # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):           # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):          # v1 * 3
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):         # 3 * v1 (reversed operands)
        return self.__mul__(scalar)

    def __eq__(self, other):            # v1 == v2
        return self.x == other.x and self.y == other.y

    def __len__(self):                  # len(v)
        return int((self.x**2 + self.y**2) ** 0.5)

    def __abs__(self):                  # abs(v)
        return (self.x**2 + self.y**2) ** 0.5

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"


v1 = Vector(2, 3)
v2 = Vector(1, 4)

print(v1 + v2)   # Vector(3, 7)   → __add__
print(v1 - v2)   # Vector(1, -1)  → __sub__
print(v1 * 3)    # Vector(6, 9)   → __mul__
print(3 * v1)    # Vector(6, 9)   → __rmul__
print(v1 == v2)  # False          → __eq__
print(abs(v1))   # 3.605...       → __abs__

# Common operator dunder methods:
# __add__  +       __radd__  right +
# __sub__  -       __mul__   *
# __truediv__ /    __floordiv__ //
# __mod__  %       __pow__   **
# __eq__   ==      __ne__    !=
# __lt__   <       __gt__    >
# __le__   <=      __ge__    >=
# __and__  &       __or__    |
# __len__  len()   __abs__   abs()
# __contains__ in  __getitem__ []

## Q17. Method Overloading in Python

In [ ]:
# Python does NOT support native method overloading
# (unlike Java/C++ where you can define same method with different signatures)

# ❌ This doesn't work like Java — last definition wins
class Bad:
    def add(self, a, b):
        return a + b

    def add(self, a, b, c):   # overwrites the previous add()!
        return a + b + c

# bad.add(1, 2)  → TypeError: add() missing 1 argument 'c'


# ✅ Python way 1 — default arguments
class Calculator:
    def add(self, a, b, c=0):
        return a + b + c

calc = Calculator()
print(calc.add(1, 2))     # 3
print(calc.add(1, 2, 3))  # 6


# ✅ Python way 2 — *args for variable arguments
class Calculator:
    def add(self, *args):
        return sum(args)

calc = Calculator()
print(calc.add(1, 2))        # 3
print(calc.add(1, 2, 3))     # 6
print(calc.add(1, 2, 3, 4))  # 10


# ✅ Python way 3 — singledispatch (true overloading by type)
from functools import singledispatch

@singledispatch
def process(data):
    raise TypeError(f"Unsupported type: {type(data)}")

@process.register(int)
def _(data):
    return f"Processing integer: {data * 2}"

@process.register(str)
def _(data):
    return f"Processing string: {data.upper()}"

@process.register(list)
def _(data):
    return f"Processing list of {len(data)} items"

print(process(5))           # Processing integer: 10
print(process("hello"))     # Processing string: HELLO
print(process([1, 2, 3]))   # Processing list of 3 items

---
# 🽟 5. Abstraction

## Q18. Abstract Classes in Python

In [ ]:
from abc import ABC, abstractmethod

# Abstract class — cannot be instantiated, defines a contract
class Shape(ABC):
    @abstractmethod
    def area(self):
        pass  # subclasses MUST implement this

    @abstractmethod
    def perimeter(self):
        pass  # subclasses MUST implement this

    def describe(self):  # concrete method — inherited as-is
        return f"Area: {self.area()}, Perimeter: {self.perimeter()}"

# shape = Shape()  # ❌ TypeError: Can't instantiate abstract class Shape

# Concrete subclass — must implement ALL abstract methods
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        import math
        return math.pi * self.radius ** 2

    def perimeter(self):
        import math
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)

c = Circle(5)
r = Rectangle(4, 6)
print(c.describe())  # Area: 78.53..., Perimeter: 31.41...
print(r.describe())  # Area: 24, Perimeter: 20

# Partial implementation — still abstract
class PartialShape(Shape):
    def area(self):
        return 0
    # perimeter() not implemented → still abstract!

# PartialShape()  # ❌ TypeError: Can't instantiate abstract class

## Q19. Abstract Class vs Interface

In [ ]:
from abc import ABC, abstractmethod

# Python doesn't have a separate 'interface' keyword (unlike Java)
# But we simulate interfaces using ABC with only abstract methods

# Abstract Class — can have:
# - Abstract methods (must be overridden)
# - Concrete methods (inherited as-is)
# - Instance variables
# - Constructor (__init__)
class AbstractAnimal(ABC):
    def __init__(self, name):       # constructor — allowed
        self.name = name

    @abstractmethod
    def speak(self): pass           # must override

    def breathe(self):              # concrete method — inherited
        return f"{self.name} is breathing"


# Interface (simulated) — ONLY abstract methods, no implementation
class Flyable(ABC):
    @abstractmethod
    def fly(self): pass             # pure contract

class Swimmable(ABC):
    @abstractmethod
    def swim(self): pass            # pure contract

# A class can implement multiple interfaces
class Duck(AbstractAnimal, Flyable, Swimmable):
    def speak(self):   return "Quack!"
    def fly(self):     return f"{self.name} is flying"
    def swim(self):    return f"{self.name} is swimming"

duck = Duck('Donald')
print(duck.speak())    # Quack!
print(duck.fly())      # Donald is flying
print(duck.swim())     # Donald is swimming
print(duck.breathe())  # Donald is breathing  ← from AbstractAnimal

# Abstract Class vs Interface summary:
# Abstract Class → partial implementation, single inheritance focus
# Interface      → pure contract, multiple inheritance ok
# Python blurs the line — both done with ABC

## Q20. `ABC`, `@abstractmethod`, and enforcement

In [ ]:
from abc import ABC, abstractmethod, abstractproperty

# ABC = Abstract Base Class (inherit from this to make class abstract)
# @abstractmethod = marks a method as abstract (must be overridden)

class Vehicle(ABC):

    @abstractmethod
    def start(self):
        pass

    @property
    @abstractmethod
    def fuel_type(self):          # abstract property
        pass

    @classmethod
    @abstractmethod
    def category(cls):            # abstract classmethod
        pass

    @staticmethod
    @abstractmethod
    def max_speed():              # abstract staticmethod
        pass

class ElectricCar(Vehicle):
    def start(self):
        return "Silent start"

    @property
    def fuel_type(self):
        return "Electric"

    @classmethod
    def category(cls):
        return "EV"

    @staticmethod
    def max_speed():
        return 250

car = ElectricCar()
print(car.start())       # Silent start
print(car.fuel_type)     # Electric
print(ElectricCar.category())   # EV
print(ElectricCar.max_speed())  # 250

# What if you DON'T implement an abstract method?
class Incomplete(Vehicle):
    def start(self):
        return "Starting"
    # forgot fuel_type, category, max_speed

# Incomplete()  # ❌ TypeError: Can't instantiate abstract class Incomplete
# with abstract methods fuel_type, category, max_speed

---
# ⚙️ 6. Dunder / Magic Methods

## Q21. Common Dunder Methods

In [ ]:
class SmartList:
    def __init__(self, items=None):      # constructor/initializer
        self.items = items or []

    def __str__(self):                   # str(obj), print(obj)
        return f"SmartList: {self.items}"

    def __repr__(self):                  # repr(obj), debug
        return f"SmartList(items={self.items!r})"

    def __len__(self):                   # len(obj)
        return len(self.items)

    def __getitem__(self, index):        # obj[index]
        return self.items[index]

    def __setitem__(self, index, value): # obj[index] = value
        self.items[index] = value

    def __delitem__(self, index):        # del obj[index]
        del self.items[index]

    def __contains__(self, item):        # item in obj
        return item in self.items

    def __iter__(self):                  # for item in obj
        return iter(self.items)

    def __add__(self, other):            # obj1 + obj2
        return SmartList(self.items + other.items)

    def __eq__(self, other):             # obj1 == obj2
        return self.items == other.items

    def __bool__(self):                  # bool(obj), if obj:
        return len(self.items) > 0


sl = SmartList([1, 2, 3])
print(sl)             # SmartList: [1, 2, 3]    → __str__
print(len(sl))        # 3                       → __len__
print(sl[0])          # 1                       → __getitem__
sl[0] = 10            #                         → __setitem__
print(2 in sl)        # True                    → __contains__
for item in sl:       #                         → __iter__
    print(item)
sl2 = SmartList([4, 5])
print(sl + sl2)       # SmartList: [10, 2, 3, 4, 5] → __add__
print(bool(sl))       # True                    → __bool__
print(bool(SmartList())) # False

## Q22. `__eq__`, `__lt__`, `__gt__` and `@functools.total_ordering`

In [ ]:
from functools import total_ordering

# Without total_ordering — must define all comparison methods manually
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def __eq__(self, other):   # ==
        return self.grade == other.grade

    def __lt__(self, other):   # <
        return self.grade < other.grade

    def __gt__(self, other):   # >
        return self.grade > other.grade

    def __le__(self, other):   # <=
        return self.grade <= other.grade

    def __ge__(self, other):   # >=
        return self.grade >= other.grade


# With @total_ordering — define only __eq__ and ONE of <, <=, >, >=
# Python auto-generates the rest!
@total_ordering
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def __eq__(self, other):
        return self.grade == other.grade

    def __lt__(self, other):          # define just this one
        return self.grade < other.grade
    # __gt__, __le__, __ge__ are auto-generated!

    def __repr__(self):
        return f"Student({self.name}, {self.grade})"


s1 = Student('Alice', 90)
s2 = Student('Bob', 75)
s3 = Student('Charlie', 90)

print(s1 > s2)   # True   → auto-generated __gt__
print(s1 < s2)   # False  → __lt__
print(s1 == s3)  # True   → __eq__
print(s1 >= s3)  # True   → auto-generated __ge__

# Enables sorting!
students = [s1, s2, s3]
print(sorted(students))  # sorted by grade
print(max(students))     # Student(Alice, 90)

## Q23. `__slots__`

In [ ]:
import sys

# Normal class — uses __dict__ to store attributes (flexible but uses more memory)
class NormalPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Class with __slots__ — fixed set of attributes, no __dict__
class SlottedPoint:
    __slots__ = ['x', 'y']  # only these attributes are allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y

np = NormalPoint(1, 2)
sp = SlottedPoint(1, 2)

# Memory comparison
print(sys.getsizeof(np.__dict__))  # ~200 bytes (has __dict__)
# sp has no __dict__ — more memory efficient

# NormalPoint — can add arbitrary attributes
np.z = 3       # ✅ works
np.color = 'red'  # ✅ works

# SlottedPoint — only x and y allowed
# sp.z = 3     # ❌ AttributeError: 'SlottedPoint' has no attribute 'z'

# Benefits of __slots__:
# 1. Lower memory usage (no per-instance __dict__)
# 2. Faster attribute access
# 3. Prevents accidental new attributes

# Useful when creating MANY instances of a class
class Particle:  # millions of particles in simulation
    __slots__ = ['x', 'y', 'z', 'mass', 'velocity']

    def __init__(self, x, y, z, mass, velocity):
        self.x = x
        self.y = y
        self.z = z
        self.mass = mass
        self.velocity = velocity

# __slots__ with inheritance:
class Point3D(SlottedPoint):
    __slots__ = ['z']  # only add NEW slots in subclass

    def __init__(self, x, y, z):
        super().__init__(x, y)
        self.z = z

## Q24. `__call__` — Making Objects Callable

In [ ]:
# __call__ lets an object be used like a function: obj()

class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):     # called when instance is used as a function
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(5))   # 10  — same as double.__call__(5)
print(triple(5))   # 15
print(callable(double))  # True


# Practical use 1 — Stateful function (remembers state between calls)
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count

counter = Counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3


# Practical use 2 — Decorator class (callable class as decorator)
class Timer:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        import time
        start = time.time()
        result = self.func(*args, **kwargs)
        print(f"{self.func.__name__} took {time.time() - start:.4f}s")
        return result

@Timer
def slow_add(a, b):
    import time
    time.sleep(0.1)
    return a + b

print(slow_add(1, 2))  # slow_add took 0.1003s → 3


# Practical use 3 — Memoization
class Memoize:
    def __init__(self, func):
        self.func = func
        self.cache = {}

    def __call__(self, *args):
        if args not in self.cache:
            self.cache[args] = self.func(*args)
        return self.cache[args]

@Memoize
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(fib(35))  # fast due to memoization

## Q25. `__new__` vs `__init__`

In [ ]:
# __new__(cls)  → creates and RETURNS a new instance (class method)
# __init__(self)→ initializes the already-created instance

class MyClass:
    def __new__(cls, *args, **kwargs):
        print(f"1. __new__ called — creating instance of {cls}")
        instance = super().__new__(cls)  # creates the object
        return instance                  # must return the instance!

    def __init__(self, value):
        print(f"2. __init__ called — initializing with value={value}")
        self.value = value

obj = MyClass(42)
# 1. __new__ called — creating instance of <class 'MyClass'>
# 2. __init__ called — initializing with value=42


# When to override __new__:

# Use case 1 — Singleton pattern
class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance  # always return the same instance

s1 = Singleton()
s2 = Singleton()
print(s1 is s2)  # True — same object!


# Use case 2 — Immutable class (like subclassing int or str)
class PositiveInt(int):
    def __new__(cls, value):
        if value <= 0:
            raise ValueError("Must be positive!")
        return super().__new__(cls, value)  # int is immutable, must use __new__

n = PositiveInt(5)
print(n)         # 5
print(n + 3)     # 8
# PositiveInt(-1) # ❌ ValueError: Must be positive!


# Key difference:
# __new__  → class method, creates object, returns instance, called FIRST
# __init__ → instance method, initializes object, returns None, called SECOND
# 99% of the time you only need __init__

---
# 🏭 7. Class & Static Methods

## Q26. `@classmethod` vs `@staticmethod` vs Instance Method

In [ ]:
class MyClass:
    class_var = 'shared'

    def __init__(self, value):
        self.value = value

    # Instance method — receives self (the instance)
    def instance_method(self):
        return f"instance: value={self.value}, class_var={self.class_var}"

    # Class method — receives cls (the class itself)
    @classmethod
    def class_method(cls):
        return f"class: class_var={cls.class_var}, class={cls.__name__}"

    # Static method — receives nothing (just a utility function)
    @staticmethod
    def static_method(x, y):
        return f"static: {x + y}"   # no access to self or cls!


obj = MyClass(42)

# Instance method — needs an instance
print(obj.instance_method())          # instance: value=42, class_var=shared

# Class method — can call on class or instance
print(MyClass.class_method())         # class: class_var=shared, class=MyClass
print(obj.class_method())             # same result

# Static method — can call on class or instance
print(MyClass.static_method(3, 4))    # static: 7
print(obj.static_method(3, 4))        # static: 7

# Summary:
# instance method → knows about the instance AND class (via self)
# classmethod     → knows about the class only (via cls)
# staticmethod    → knows about neither — just a namespaced function

## Q27. `@classmethod` as Alternative Constructor

In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    # Alternative constructor — from string
    @classmethod
    def from_string(cls, date_string):
        year, month, day = map(int, date_string.split('-'))
        return cls(year, month, day)   # creates a new instance via cls()

    # Alternative constructor — today's date
    @classmethod
    def today(cls):
        import datetime
        today = datetime.date.today()
        return cls(today.year, today.month, today.day)

    # Alternative constructor — from timestamp
    @classmethod
    def from_timestamp(cls, timestamp):
        import datetime
        dt = datetime.datetime.fromtimestamp(timestamp)
        return cls(dt.year, dt.month, dt.day)

    def __repr__(self):
        return f"Date({self.year}, {self.month}, {self.day})"


# Multiple ways to create a Date
d1 = Date(2024, 1, 15)              # standard
d2 = Date.from_string('2024-01-15') # from string
d3 = Date.today()                   # today

print(d1)   # Date(2024, 1, 15)
print(d2)   # Date(2024, 1, 15)
print(d3)   # Date(2026, 3, 26)  ← today


# Works with inheritance!
class ExtendedDate(Date):
    def __init__(self, year, month, day, timezone='UTC'):
        super().__init__(year, month, day)
        self.timezone = timezone

# cls in from_string() will be ExtendedDate, not Date
# So the alternative constructor returns an ExtendedDate instance!
ed = ExtendedDate.from_string('2024-01-15')
print(type(ed))  # <class '__main__.ExtendedDate'>

## Q28. `cls` vs `self`

In [ ]:
class Animal:
    count = 0               # class variable

    def __init__(self, name):
        self.name = name     # self → refers to THIS INSTANCE
        Animal.count += 1

    def speak(self):
        # self → the specific instance calling this method
        return f"{self.name} speaks"  # self.name is THIS animal's name

    @classmethod
    def get_count(cls):
        # cls → the CLASS itself (not an instance)
        return f"{cls.__name__} count: {cls.count}"

    @classmethod
    def create(cls, name):
        # cls() creates a new instance of whichever class called this
        return cls(name)  # works correctly in subclasses too!

class Dog(Animal):
    def speak(self):
        return f"{self.name} says Woof!"

a = Animal('Generic')
d1 = Dog('Rex')
d2 = Dog.create('Buddy')   # cls = Dog → returns Dog instance

print(Animal.get_count())   # Animal count: 3  ← cls.count
print(Dog.get_count())      # Dog count: 3     ← cls = Dog, but count is shared
print(type(d2))             # <class '__main__.Dog'>

# self → instance reference, available in instance methods
# cls  → class reference, available in class methods
# Both are just conventions — you could name them anything (but don't!)

---
# 🎨 8. Design Patterns & Advanced

## Q29. Composition vs Inheritance

In [ ]:
# Inheritance = "IS-A" relationship
# Composition = "HAS-A" relationship

# ❌ Inheritance — wrong use (Car IS-A Engine? No!)
class Engine:
    def start(self):
        return "Engine started"

class BadCar(Engine):  # Car IS-A Engine? Wrong!
    pass


# ✅ Composition — Car HAS-A Engine
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        return f"Engine ({self.horsepower}hp) started"

class GPS:
    def navigate(self, destination):
        return f"Navigating to {destination}"

class MusicSystem:
    def play(self, song):
        return f"Playing {song}"

class Car:   # Car HAS-A Engine, GPS, MusicSystem
    def __init__(self, brand, horsepower):
        self.brand = brand
        self.engine = Engine(horsepower)   # composition
        self.gps = GPS()                   # composition
        self.music = MusicSystem()         # composition

    def start(self):
        return f"{self.brand}: {self.engine.start()}"

car = Car('BMW', 300)
print(car.start())                   # BMW: Engine (300hp) started
print(car.gps.navigate('Airport'))   # Navigating to Airport
print(car.music.play('Jazz'))        # Playing Jazz

# Prefer composition over inheritance because:
# ✅ More flexible — swap components at runtime
# ✅ Avoids deep inheritance chains
# ✅ Easier to test (mock individual components)
# ✅ No unexpected method inheritance

# When to use inheritance:
# ✅ Genuine IS-A relationship (Dog IS-A Animal)
# ✅ Want to override specific behavior
# ✅ Using abstract base classes to define a contract

## Q30. Mixins

In [ ]:
# Mixin = small, reusable class that adds a specific capability
# NOT meant to be instantiated on its own
# Combined via multiple inheritance

class JSONMixin:
    def to_json(self):
        import json
        return json.dumps(self.__dict__)

    @classmethod
    def from_json(cls, json_str):
        import json
        data = json.loads(json_str)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj

class LogMixin:
    def log(self, message):
        print(f"[{self.__class__.__name__}] {message}")

class ValidateMixin:
    def validate(self):
        for attr, value in self.__dict__.items():
            if value is None:
                raise ValueError(f"{attr} cannot be None")
        return True

class CompareMixin:
    def __eq__(self, other):
        return self.__dict__ == other.__dict__


# Apply mixins to any class
class User(JSONMixin, LogMixin, ValidateMixin, CompareMixin):
    def __init__(self, name, email):
        self.name = name
        self.email = email

class Product(JSONMixin, LogMixin, ValidateMixin):
    def __init__(self, name, price):
        self.name = name
        self.price = price


user = User('Alice', 'alice@example.com')
user.log("User created")         # [User] User created
print(user.to_json())            # {"name": "Alice", "email": "alice@example.com"}
print(user.validate())           # True

product = Product('Laptop', 999)
product.log("Product added")     # [Product] Product added
print(product.to_json())         # {"name": "Laptop", "price": 999}

# Mixin naming convention: usually ends with 'Mixin'
# Mixins vs regular inheritance:
# Mixin         → adds capabilities (can do something)
# Base class    → defines identity (is something)

## Q31. Metaclasses and `type`

In [ ]:
# In Python, EVERYTHING is an object — even classes!
# A metaclass is the class of a class
# type is the default metaclass of all classes

class Dog:
    pass

print(type(Dog))        # <class 'type'> — Dog's class is type
print(type(Dog()))      # <class '__main__.Dog'>
print(type(42))         # <class 'int'>
print(type(int))        # <class 'type'>

# Creating a class dynamically with type()
# type(name, bases, dict)
Cat = type('Cat', (object,), {
    'sound': 'Meow',
    'speak': lambda self: self.sound
})

cat = Cat()
print(cat.speak())  # Meow


# Custom Metaclass — intercepts class creation
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=SingletonMeta):
    def __init__(self):
        self.connection = "Connected"

db1 = Database()
db2 = Database()
print(db1 is db2)  # True — same instance!


# Metaclass to enforce method naming convention
class EnforceLowercase(type):
    def __new__(mcs, name, bases, dct):
        for attr in dct:
            if not attr.startswith('_') and attr != attr.lower():
                raise TypeError(f"Method '{attr}' must be lowercase")
        return super().__new__(mcs, name, bases, dct)

class GoodClass(metaclass=EnforceLowercase):
    def hello(self): pass   # ✅ lowercase

# class BadClass(metaclass=EnforceLowercase):
#     def Hello(self): pass  # ❌ TypeError: Method 'Hello' must be lowercase

# When to use metaclasses:
# - ORM frameworks (Django models use metaclasses)
# - API validation
# - Enforcing coding standards
# - Most times: use class decorators instead (simpler)

## Q32. Singleton Pattern

In [ ]:
# Singleton = only ONE instance of a class can exist

# Method 1 — Using __new__
class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

s1 = Singleton()
s2 = Singleton()
print(s1 is s2)   # True
print(id(s1) == id(s2))  # True


# Method 2 — Using metaclass
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self):
        self.debug = False
        self.db_url = 'postgresql://localhost/mydb'


# Method 3 — Using decorator (clean, simple)
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Logger:
    def __init__(self):
        self.logs = []

    def log(self, message):
        self.logs.append(message)

log1 = Logger()
log2 = Logger()
log1.log('Hello')
print(log2.logs)   # ['Hello'] — same instance!


# Method 4 — Module-level instance (Pythonic way)
# Just create an instance at module level in a separate file
# config.py:
# class _Config:
#     debug = False
# config = _Config()  # everyone imports this single instance

# Real-world uses: DB connections, Config objects, Logger, Thread pools

## Q33. `@dataclass`

In [ ]:
from dataclasses import dataclass, field, asdict, astuple

# Regular class — lots of boilerplate
class RegularPoint:
    def __init__(self, x, y, z=0):
        self.x = x
        self.y = y
        self.z = z

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y}, z={self.z})"

    def __eq__(self, other):
        return (self.x, self.y, self.z) == (other.x, other.y, other.z)


# @dataclass — auto-generates __init__, __repr__, __eq__
@dataclass
class Point:
    x: float
    y: float
    z: float = 0.0   # default value

p1 = Point(1, 2)
p2 = Point(1, 2, 0)
print(p1)           # Point(x=1, y=2, z=0.0)   → auto __repr__
print(p1 == p2)     # True                      → auto __eq__


# Advanced dataclass options
@dataclass(order=True, frozen=True)  # frozen=immutable, order=comparison operators
class Student:
    grade: int
    name: str
    # order=True → auto-generates __lt__, __gt__, etc. based on field order
    # frozen=True → immutable (like a tuple), generates __hash__

s1 = Student(90, 'Alice')
s2 = Student(75, 'Bob')
print(s1 > s2)             # True  → auto-generated comparison
# s1.grade = 95            # ❌ FrozenInstanceError — immutable!
print(hash(s1))            # works because frozen=True generates __hash__


# Mutable default — use field(default_factory=)
@dataclass
class Team:
    name: str
    members: list = field(default_factory=list)  # ✅ correct
    # members: list = []  # ❌ shared mutable default!

    def add_member(self, member):
        self.members.append(member)

t1 = Team('Alpha')
t2 = Team('Beta')
t1.add_member('Alice')
print(t2.members)  # [] — independent lists!


# Utility functions
p = Point(3, 4, 5)
print(asdict(p))    # {'x': 3, 'y': 4, 'z': 5}
print(astuple(p))   # (3, 4, 5)

# @dataclass vs regular class:
# @dataclass → less boilerplate, auto __init__/__repr__/__eq__, type hints
# Regular    → more control, custom logic in __init__, no type hint requirement

## Q34. `__init_subclass__`

In [ ]:
# __init_subclass__ is called whenever the class is subclassed
# Lets you hook into the subclass creation process

# Use case 1 — Auto-register subclasses (plugin system)
class Plugin:
    _registry = {}

    def __init_subclass__(cls, plugin_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        name = plugin_name or cls.__name__.lower()
        Plugin._registry[name] = cls
        print(f"Registered plugin: {name} → {cls}")

class PDFExporter(Plugin, plugin_name='pdf'):
    def export(self): return "Exporting to PDF"

class CSVExporter(Plugin, plugin_name='csv'):
    def export(self): return "Exporting to CSV"

class XMLExporter(Plugin):           # uses class name 'xmlexporter'
    def export(self): return "Exporting to XML"

# Output:
# Registered plugin: pdf → PDFExporter
# Registered plugin: csv → CSVExporter
# Registered plugin: xmlexporter → XMLExporter

print(Plugin._registry)

# Use registry to get plugin by name
exporter = Plugin._registry['pdf']()
print(exporter.export())  # Exporting to PDF


# Use case 2 — Enforce required attributes on subclasses
class Model:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if not hasattr(cls, 'table_name'):
            raise TypeError(f"{cls.__name__} must define 'table_name'")

class UserModel(Model):
    table_name = 'users'     # ✅ required attribute present

# class BadModel(Model):     # ❌ TypeError: BadModel must define 'table_name'
#     pass


# Use case 3 — Validate subclass methods
class Service:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if not hasattr(cls, 'handle'):
            raise TypeError(f"{cls.__name__} must implement handle()")

class EmailService(Service):
    def handle(self, data):
        return f"Sending email: {data}"

# __init_subclass__ vs metaclass:
# __init_subclass__ → simpler, cleaner, Python 3.6+
# metaclass         → more powerful but complex

---
# 🎉 Summary

| # | Topic | Key Questions |
|---|-------|---------------|
| ✅ 1 | Classes & Objects | Class vs Object, `__init__`, instance vs class vars, `self`, `__str__` vs `__repr__` |
| ✅ 2 | Encapsulation | Public/Protected/Private, name mangling, `@property` |
| ✅ 3 | Inheritance | Single/Multiple, Diamond problem, MRO, `super()`, `isinstance`, overriding |
| ✅ 4 | Polymorphism | Polymorphism, duck typing, operator overloading, method overloading |
| ✅ 5 | Abstraction | Abstract classes, ABC, `@abstractmethod`, abstract vs interface |
| ✅ 6 | Dunder Methods | Magic methods, `__eq__`/`__lt__`, `__slots__`, `__call__`, `__new__` vs `__init__` |
| ✅ 7 | Class & Static Methods | `@classmethod`, `@staticmethod`, instance methods, `cls` vs `self` |
| ✅ 8 | Design Patterns | Composition vs Inheritance, Mixins, Metaclasses, Singleton, `@dataclass`, `__init_subclass__` |